# TAHAP 2

### Subbab 2.1 — Import pustaka

Memuat library yang dibutuhkan untuk tahap ini.


In [10]:
import os
import pandas as pd
import re
# from google.colab import drive
import nltk # Using NLTK for tokenization to count words

### Subbab 2.2 — Konfigurasi path dan parameter

Mengatur path, parameter, dan variabel penting sebelum proses dijalankan.


In [11]:
# --- Bagian Konfigurasi ---
BASE_DRIVE_PATH = ".." # Changed from colab path to local relative path

# Path untuk input data dari Notebook 1
PATH_RAW_TEXT_INPUT = os.path.join(BASE_DRIVE_PATH, "data/raw")
PATH_INITIAL_SCRAPER_CSV_INPUT = os.path.join(BASE_DRIVE_PATH, "Scraper_CSVs")

# Path untuk output processed data
PATH_PROCESSED_OUTPUT = os.path.join(BASE_DRIVE_PATH, "data/processed")
os.makedirs(PATH_PROCESSED_OUTPUT, exist_ok=True)

# --- Pengaturan NLTK (untuk menghitung jumlah kata) ---
try:
    nltk.data.find('tokenizers/punkt')
  # Add this line ke also check dan download 'punkt_tab'
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    print("NLTK 'punkt' or 'punkt_tab' not found. Downloading...")
    nltk.download('punkt')
    nltk.download('punkt_tab') # Download the missing resource
    print("NLTK 'punkt' and 'punkt_tab' downloaded.")
except Exception as e:
     print(f"An unexpected error occurred during NLTK setup: {e}")

NLTK 'punkt' or 'punkt_tab' not found. Downloading...


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


NLTK 'punkt' and 'punkt_tab' downloaded.


### Subbab 2.3 — Fungsi bantu

Berisi fungsi utilitas yang dipakai ulang pada proses inti.


In [12]:
# --- Fungsi Bantu ---

def load_latest_scraper_csv(csv_folder_path):
    """Memuat file CSV terbaru dari folder yang ditentukan."""
    try:
        csv_files = [f for f in os.listdir(csv_folder_path) if f.endswith('.csv')]
        if not csv_files:
            print(f"No CSV files found in {csv_folder_path}")
            return None
    # Cari file terbaru berdasarkan pola nama file (jika memuat tanggal) atau waktu modifikasi
    # Assuming filenames might include dates like 'putusan_ma_KEYWORD_YYYY-MM-DD.csv'
        csv_files.sort(key=lambda name: os.path.getmtime(os.path.join(csv_folder_path, name)), reverse=True)
        latest_csv_filename = csv_files[0]
        print(f"Loading latest scraper CSV: {latest_csv_filename}")
        return pd.read_csv(os.path.join(csv_folder_path, latest_csv_filename))
    except Exception as e:
        print(f"Error loading latest CSV: {e}")
        return None

def read_raw_text_file(filename, raw_text_folder):
    """Membaca isi file teks mentah."""
    filepath = os.path.join(raw_text_folder, filename)
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            return f.read()
    except FileNotFoundError:
        print(f"Warning: Raw text file not found: {filepath}")
        return ""
    except Exception as e:
        print(f"Error reading raw text file {filepath}: {e}")
        return ""

def count_words(text):
    """Counts words in a teks menggunakan NLTK tokenization."""
    if pd.isna(text) or not text:
        return 0
    tokens = nltk.word_tokenize(str(text))
    return len(tokens)

def extract_section_heuristic(text, keywords_start, keywords_end=None, limit_chars=5000):
    """
    Extracts a section of text based on start and optional end keywords using heuristics.
    keywords_start: list of regex patterns to find the start of the section.
    keywords_end: list of regex patterns to find the end of the section (optional).
    """
    if pd.isna(text) or not text:
        return ""

    text_lower = text.lower() # Search in lowercase for broader matching
    start_index = -1

  # Cari indeks awal
    for kw_pattern in keywords_start:
        match = re.search(kw_pattern, text_lower)
        if match:
            start_index = match.end() # Start after the keyword
            break

    if start_index == -1:
        return "" # Start keyword not found

  # Cari indeks akhir jika keywords_end disediakan
    end_index = len(text) # Default to end of text
    if keywords_end:
    # Cari kata kunci akhir *setelah* start_index
    # Agar akurat, pencarian dilakukan pada potongan teks asli
        min_end_index_found = len(text)
        found_end_keyword = False
        for kw_pattern in keywords_end:
      # Cari pada bagian teks *setelah* start_index
            match = re.search(kw_pattern, text_lower[start_index:])
            if match:
        # match.start() is relative ke text_lower[start_index:]
        # Add start_index ke get absolute position in original text_lower
                current_end_match_pos = start_index + match.start()
                if current_end_match_pos < min_end_index_found:
                    min_end_index_found = current_end_match_pos
                    found_end_keyword = True
        if found_end_keyword:
            end_index = min_end_index_found

    extracted_text = text[start_index:end_index].strip()
    return extracted_text[:limit_chars] # Limit character length if needed

### Subbab 2.4 — Proses inti

Bagian ini menjalankan:  Main Processing Logic .


In [13]:
log_cleaning_action("=" * 60)
log_cleaning_action("TAHAP 2: Case Representation — MULAI")
log_cleaning_action("=" * 60)
print("Starting Tahap 2: Case Representation")

# 1. Muat initial data (CSV dari scraper dan raw teks files)
log_cleaning_action("[Tahap 2.1] Loading Data dari CSV dan raw text files...")
print(f"\n[1. Loading Data]")
df_initial = load_latest_scraper_csv(PATH_INITIAL_SCRAPER_CSV_INPUT)

if df_initial is None or df_initial.empty:
    print("Could not load initial scraper CSV. Please ensure Notebook 1 ran successfully and the CSV exists.")
  # Exit or handle error appropriately
else:
    log_cleaning_action(f"[Tahap 2.1] Loaded {len(df_initial)} records dari CSV.")
    print(f"Loaded initial data with {len(df_initial)} records.")
  # Ensure 'nama_file_raw_text' column exists
    if 'nama_file_raw_text' not in df_initial.columns:
        print("ERROR: 'nama_file_raw_text' column not found in the CSV. This column is needed to load full text.")
    # Exit or handle
    else:
    # Muat full teks untuk each kasus
        df_initial['full_text_putusan'] = df_initial['nama_file_raw_text'].apply(
            lambda x: read_raw_text_file(x, PATH_RAW_TEXT_INPUT) if pd.notna(x) else ""
        )
        log_cleaning_action(f"[Tahap 2.1] Full text loaded ke DataFrame untuk {len(df_initial)} kasus.")
        print("Full text loaded into DataFrame.")

    # Initialize new columns untuk extracted features
        df_initial['ringkasan_fakta'] = ""
        df_initial['argumen_hukum_utama'] = ""
        df_initial['pasal_digunakan_extracted'] = "" # For refined pasal extraction
        df_initial['pihak_terlibat_extracted'] = ""   # For refined pihak extraction


    # 2. Metadata Extraction & Refinement
    # Sebagian besar metadata (nomor perkara, tanggal, jenis perkara) sudah ada di df_initial dari scraper.
    # Kita bisa memperhalus 'pasal_digunakan' dan 'nama_pihak' jika hasil scrape awal masih dasar.
        log_cleaning_action("[Tahap 2.2] Extracting/Refining Metadata & Text Features...")
        print(f"\n[2. Extracting/Refining Metadata & Text Features]")

    # Keywords untuk extracting "Ringkasan Fakta"
    # These are examples dan might need adjustment based on actual document structures
        fakta_keywords_start = [
            r"tentang pokok sengketa pengajuan peninjauan kembali",
            r"alasan dan penjelasan permohonan banding",
            r"alasan permohonan banding",
            r"pokok sengketa;",
            r"kronologis sengketa pajak",
            r"menimbang,\s*bahwa\s*terdakwa\s*diajukan\s*ke\s*persidangan",
            r"menimbang,\s*bahwa\s*penggugat\s*dalam\s*surat\s*gugatannya",
            r"duduk\s*perkara:",
            r"fakta-fakta\s*hukum\s*yang\s*terungkap",
            r"menimbang,\s*bahwa\s*untuk\s*membuktikan\s*dalil-dalilnya",
            r"uraian\s*singkat\s*mengenai\s*kejadian",
            r"tentang\s*duduk\s*perkara"
        ]
        fakta_keywords_end = [ # Stop before legal considerations or verdict
            r"pertimbangan hukum",
            r"tentang pertimbangan hukum",
            r"menimbang, bahwa terhadap alasan-alasan peninjauan kembali",
            r"menimbang,\s*bahwa\s*selanjutnya\s*majelis\s*hakim\s*akan\s*mempertimbangkan",
            r"pertimbangan\s*hukum",
            r"tentang\s*pertimbangan\s*hukum",
            r"amar\s*putusan",
            r"mengadili"
        ]

    # Keywords untuk "Argumen Hukum Utama" (Pertimbangan Hukum)
        argumen_keywords_start = [
            r"pertimbangan hukum", # Header utama
            r"menimbang, bahwa terhadap alasan-alasan peninjauan kembali tersebut, mahkamah agung berpendapat",
            r"menimbang, bahwa alasan-alasan permohonan pemohon peninjauan kembali tidak dapat dibenarkan",
            r"menimbang, bahwa alasan-alasan permohonan pemohon peninjauan kembali dapat dibenarkan",
            r"pertimbangan\s*hukum",
            r"tentang\s*pertimbangan\s*hukum",
            r"menimbang,\s*bahwa\s*terhadap\s*eksepsi", # Start of legal reasoning
            r"menimbang,\s*bahwa\s*majelis\s*hakim\s*berpendapat",
            r"menimbang,\s*bahwa\s*oleh\s*karena\s*itu\s*dengan\s*memperhatikan"
        ]
        argumen_keywords_end = [ # Stop before the final verdict/amar
            r"memperhatikan pasal-pasal dari undang-undang",
            r"mengadili,",
            r"amar\s*putusan",
            r"mengadili",
            r"memutuskan",
            r"menetapkan"
        ]

    # Regex untuk "Pasal Digunakan" (example, very basic, often complex)
    # Looks untuk patterns like "Pasal X ayat (Y) UU No. Z Tahun A" or KUHP/KUHAP etc.
        pasal_regex_patterns = [
            r"pasal\s*\d+\s*(ayat\s*\(?\s*\d+\s*\)?\s*)?(huruf\s*[a-z]\s*)?\s*(uu|undang-undang)\s*(nomor|no\.)?\s*\d+\s*tahun\s*\d+",
            r"pasal\s*\d+\s*(ayat\s*\(?\s*\d+\s*\)?\s*)?\s*kuhp(?:idana)?",
            r"pasal\s*\d+\s*(ayat\s*\(?\s*\d+\s*\)?\s*)?\s*kuhperdata",
            r"peraturan pemerintah\s*(nomor|no\.)?\s*\d+\s*tahun\s*\d+",
            r"peraturan menteri keuangan\s*(nomor|no\.)?[\s\w./-]+", # Mencakup format seperti 78/PMK.03/2010
            r"keputusan direktur jenderal pajak\s*(nomor|no\.)?[\s\w./-]+", # Mencakup format seperti KEP-539/PJ./2001
            r"surat edaran\s*(direktur jenderal pajak)?\s*(nomor|no\.)?[\s\w./-]+" # Mencakup format seperti SE-90/PJ/2011
        ]

        for index, row in df_initial.iterrows():
            full_text = str(row['full_text_putusan'])
            full_text_lower = full_text.lower() # Gunakan versi lowercase untuk pencarian

            log_cleaning_action(f"[Tahap 2.2] Processing case_id: {row.get('case_id', 'N/A')}")
            print(f"Processing case_id: {row.get('case_id', 'N/A')}...")

      # Ekstraksi Pihak Terlibat (Lebih Akurat)
      # Pola: PEMOHON... melawan: TERMOHON...
            pihak_match = re.search(
                r"(pemohon peninjauan kembali.*?)(?:melawan:|lawan)(.*?)(?:mahkamah agung tersebut;)",
                full_text,
                re.IGNORECASE | re.DOTALL
            )
            if pihak_match:
                pemohon_text = re.sub(r'\s+', ' ', pihak_match.group(1)).strip()
                termohon_text = re.sub(r'\s+', ' ', pihak_match.group(2)).strip()
                df_initial.loc[index, 'pihak_terlibat_extracted'] = f"Pemohon: {pemohon_text} vs Termohon: {termohon_text}"
            else:
        # Fallback jika pola utama tidak ditemukan
                df_initial.loc[index, 'pihak_terlibat_extracted'] = row.get('nama_pihak', 'N/A')


      # Ekstrak Ringkasan Fakta (menggunakan keywords baru)
            df_initial.loc[index, 'ringkasan_fakta'] = extract_section_heuristic(
                full_text, fakta_keywords_start, fakta_keywords_end, limit_chars=4000
            )

      # Ekstrak Argumen Hukum Utama (menggunakan keywords baru)
            df_initial.loc[index, 'argumen_hukum_utama'] = extract_section_heuristic(
                full_text, argumen_keywords_start, argumen_keywords_end, limit_chars=5000
            )

      # Ekstrak Pasal Digunakan (menggunakan regex baru)
            found_pasal_list = []
      # Cari di seluruh teks (versi lowercase)
            for pattern in pasal_regex_patterns:
                matches = re.findall(pattern, full_text_lower)
                for match in matches:
                    pasal_text = match if isinstance(match, str) else " ".join(filter(None, match))
                    normalized_pasal = ' '.join(pasal_text.split()).strip()
                    if normalized_pasal and normalized_pasal not in found_pasal_list:
                        found_pasal_list.append(normalized_pasal)

      # Gabungkan hasil dan bersihkan dari duplikat
            df_initial.loc[index, 'pasal_digunakan_extracted'] = "; ".join(sorted(list(set(found_pasal_list)))) if found_pasal_list else row.get('pasal_digunakan', '')

      # Ekstrak Pihak Terlibat (refined) - Also complex
      # Example: look untuk "antara ... sebagai Penggugat; dan ... sebagai Tergugat"
      # Ini butuh regex yang lebih spesifik domain. Untuk saat ini, gunakan 'nama_pihak' dari hasil scrape awal.
      # or provide a placeholder jika it needs significant improvement.
      # Jika 'nama_pihak' dari scraper is good enough:
      # df_initial.loc[index, 'pihak_terlibat_extracted'] = row.get('nama_pihak', 'N/A')
      # Add more sophisticated regex here jika needed, e.g.:
      # match_pihak = re.search(r"antara\s+(.*?)\s*sebagai\s*Pemohon\s*;\s*melawan\s+(.*?)\s*sebagai\s*Termohon", full_text, re.IGNORECASE | re.DOTALL)
      # jika match_pihak:
      #  df_initial.loc[index, 'pihak_terlibat_extracted'] = f"Pemohon: {match_pihak.group(1).strip()} vs Termohon: {match_pihak.group(2).strip()}"


    # 3. Feature Engineering
        log_cleaning_action("[Tahap 2.3] Performing Feature Engineering (word counts, BoW, QA-pairs)...")
        print(f"\n[3. Performing Feature Engineering]")

    # Calculate length (jumlah kata) untuk key teks fields
        df_initial['jumlah_kata_full_text'] = df_initial['full_text_putusan'].apply(count_words)
        df_initial['jumlah_kata_ringkasan_fakta'] = df_initial['ringkasan_fakta'].apply(count_words)
        df_initial['jumlah_kata_argumen_hukum'] = df_initial['argumen_hukum_utama'].apply(count_words)
        log_cleaning_action("[Tahap 2.3] Word counts berhasil dihitung.")
        print("Word counts calculated.")

    # Bag-of-Words (BoW) - Will be implicitly handled by TF-IDF in Tahap 3.
    # Untuk this stage, kita can note its conceptual presence or skip explicit generation
    # ke avoid large sparse matrices in this intermediate CSV.
    # Jika needed, one could tokenize dan store counts, but it's often not stored directly.
        print("Conceptual Bag-of-Words representation will be handled in later stages (e.g., TF-IDF).")

    # QA-pairs sederhana - This is an advanced feature.
    # Untuk a "sederhana" system, this could be:
    # - Placeholder: Indicating it's a potential future enhancement.
    # - Heuristic: Extracting questions dari "Pertimbangan Hukum" jika any explicit questions are posed.
    # Untuk saat ini, kita akan add a placeholder column.
        df_initial['qa_pairs_sederhana'] = "NOT_IMPLEMENTED" # Placeholder
        print("QA-pairs (sederhana) marked as NOT_IMPLEMENTED (advanced feature).")


    # 4. Prepare Final DataFrame dan Simpan
        log_cleaning_action("[Tahap 2.4] Preparing and Saving Processed Data...")
        print(f"\n[4. Preparing and Saving Processed Data]")

    # Select dan rename columns ke match PDF example where possible
    # "case_id", "no_perkara", "tanggal", "ringkasan_fakta", "pasal", "pihak", "text_full"
        df_processed = df_initial.rename(columns={
            'nomor_perkara': 'no_perkara',
            'tanggal_putusan': 'tanggal', # Assuming tanggal_putusan is the main date
            'pasal_digunakan_extracted': 'pasal', # Using the extracted/refined one
            'pihak_terlibat_extracted': 'pihak',   # Using the extracted/refined one
            'full_text_putusan': 'text_full' # Full text is important
        })

    # Ensure all required columns dari PDF example are present, add jika missing
        required_cols = ["case_id", "no_perkara", "tanggal", "ringkasan_fakta", "pasal", "pihak", "text_full"]
        for col in required_cols:
            if col not in df_processed.columns:
                df_processed[col] = df_initial.get(col, pd.NA) # Get from original if renamed, else NA

    # Add other valuable columns (metadata dari scraper, engineered features)
    # Keep original 'jenis_perkara', 'amar_putusan', etc.
    # Keep word counts
        additional_cols_to_keep = [
            'judul_putusan', 'jenis_perkara', 'tingkat_proses', 'kata_kunci',
            'tahun_dokumen', 'tanggal_register', 'lembaga_peradilan', 'amar_putusan',
            'link_sumber', 'link_pdf', 'nama_file_pdf', 'nama_file_raw_text',
            'jumlah_kata_full_text', 'jumlah_kata_ringkasan_fakta', 'jumlah_kata_argumen_hukum',
            'argumen_hukum_utama', # Retain this important feature
            'qa_pairs_sederhana'
        ]

        final_columns_ordered = required_cols + [col for col in additional_cols_to_keep if col in df_processed.columns and col not in required_cols]
    # Ensure no duplicate columns dan all are present
        final_columns_ordered = sorted(list(set(final_columns_ordered)), key=final_columns_ordered.index)
        df_processed = df_processed[final_columns_ordered]


    # ── Buat kolom amar_kategori (dibutuhkan oleh Tahap 4 & 5) ──
        log_cleaning_action("[Tahap 2.3.5] Membuat kolom amar_kategori...")
        print(f"\n[3.5 Membuat kolom amar_kategori...]")

        def get_kategori_amar_inline(full_text):
            import re
            if not isinstance(full_text, str) or not full_text:
                return "Teks Tidak Valid"
            m = re.search(
                r"MENGADILI,?\s*(.*?)(Demikianlah diputuskan|Memperhatikan pasal-pasal)",
                full_text, re.IGNORECASE | re.DOTALL
            )
            if not m:
                return "Amar Tidak Ditemukan"
            t = m.group(1).lower()
            if 'menolak permohonan' in t:
                return "Menolak permohonan"
            elif 'mengabulkan seluruhnya' in t or 'mengabulkan permohonan banding' in t:
                return "Mengabulkan seluruhnya"
            elif 'mengabulkan sebagian' in t:
                return "Mengabulkan sebagian"
            elif 'tidak dapat diterima' in t:
                return "Tidak dapat diterima"
            else:
                return "Lain-lain (Perlu Cek Manual)"

        text_col_for_amar = 'text_full' if 'text_full' in df_processed.columns else 'full_text_putusan'
        if text_col_for_amar in df_processed.columns:
            df_processed['amar_kategori'] = df_processed[text_col_for_amar].apply(get_kategori_amar_inline)
            dist = df_processed['amar_kategori'].value_counts()
            print("Distribusi amar_kategori:")
            print(dist.to_string())
        else:
            df_processed['amar_kategori'] = "Teks Tidak Valid"
            print("⚠️  Kolom teks tidak ditemukan untuk membuat amar_kategori.")

    # Simpan ke CSV
        processed_csv_filename = "cases_processed.csv"
        processed_csv_filepath = os.path.join(PATH_PROCESSED_OUTPUT, processed_csv_filename)
        df_processed.to_csv(processed_csv_filepath, index=False, encoding='utf-8')
        log_cleaning_action(f"[Tahap 2.4] Processed CSV disimpan ke: {processed_csv_filepath}")
        print(f"Processed data saved to: {processed_csv_filepath}")

    # Simpan ke JSON (opsional, sesuai PDF)
        processed_json_filename = "cases_processed.json"
        processed_json_filepath = os.path.join(PATH_PROCESSED_OUTPUT, processed_json_filename)
        df_processed.to_json(processed_json_filepath, orient='records', indent=4, force_ascii=False)
        log_cleaning_action(f"[Tahap 2.4] Processed JSON disimpan ke: {processed_json_filepath}")
        print(f"Processed data also saved to: {processed_json_filepath}")

    # Simpan ke XLSX
        processed_xlsx_filename = "cases_processed.xlsx"
        processed_xlsx_filepath = os.path.join(PATH_PROCESSED_OUTPUT, processed_xlsx_filename)
        df_processed.to_excel(processed_xlsx_filepath, index=False, engine='openpyxl')
        log_cleaning_action(f"[Tahap 2.4] Processed XLSX disimpan ke: {processed_xlsx_filepath}")
        print(f"Processed data also saved to: {processed_xlsx_filepath}")

        print("\n--- Sample of Processed Data ---")
        display(df_processed.head())
        print(f"\nColumns in processed DataFrame: {df_processed.columns.tolist()}")
        print(f"Shape of processed DataFrame: {df_processed.shape}")

log_cleaning_action(f"[Tahap 2] Shape final DataFrame: {df_processed.shape if 'df_processed' in dir() else 'N/A'}")
log_cleaning_action("TAHAP 2: Case Representation — SELESAI")
print("\nTahap 2: Case Representation - Complete.")

TAHAP 2: Case Representation — MULAI
Starting Tahap 2: Case Representation
[Tahap 2.1] Loading Data dari CSV dan raw text files...

[1. Loading Data]
Loading latest scraper CSV: putusan_ma_Pajak_2026-06-04.csv
[Tahap 2.1] Loaded 43 records dari CSV.
Loaded initial data with 43 records.
[Tahap 2.1] Full text loaded ke DataFrame untuk 43 kasus.
Full text loaded into DataFrame.
[Tahap 2.2] Extracting/Refining Metadata & Text Features...

[2. Extracting/Refining Metadata & Text Features]
[Tahap 2.2] Processing case_id: case_001
Processing case_id: case_001...
[Tahap 2.2] Processing case_id: case_002
Processing case_id: case_002...
[Tahap 2.2] Processing case_id: case_003
Processing case_id: case_003...
[Tahap 2.2] Processing case_id: case_004
Processing case_id: case_004...
[Tahap 2.2] Processing case_id: case_005
Processing case_id: case_005...
[Tahap 2.2] Processing case_id: case_006
Processing case_id: case_006...
[Tahap 2.2] Processing case_id: case_007
Processing case_id: case_007...


,case_id,no_perkara,tanggal,ringkasan_fakta,pasal,pihak,text_full,judul_putusan,jenis_perkara,tingkat_proses,...,link_sumber,link_pdf,nama_file_pdf,nama_file_raw_text,jumlah_kata_full_text,jumlah_kata_ringkasan_fakta,jumlah_kata_argumen_hukum,argumen_hukum_utama,qa_pairs_sederhana,amar_kategori
0,case_001,1111/B/PK/PJK/2017 PUTUSAN Nomor 1111/B/PK/PJK...,15 Agustus 2014,,huruf e undang-undang nomor; nomor; undang-und...,Pemohon: Pemohon Peninjauan Kembali dahulu Pem...,Direktori Putusan Mahkamah Agung Republik Indo...,putusan_1111_b_pk_pjk_2017_20250609100401,Pajak,NaN,...,MANUAL_DOWNLOAD,/content/drive/MyDrive/Penalaran Komputer/Penu...,putusan_1111_b_pk_pjk_2017_20250609100401.pdf,case_001_putusan_1111_b_pk_pjk_2017_2025060910...,3649,0,722,sebagai berikut: Bahwa dalam Pasal 1 angka 1 h...,NOT_IMPLEMENTED,Lain-lain (Perlu Cek Manual)
1,case_002,1227/B/PK/PJK/2017 PUTUSAN Nomor 1227/B/PK/PJK...,5 Agustus 2016,Bahwa yang menjadi pokok sengketa dalam permoh...,ayat (1) undang-undang nomor; direktur jendera...,Pemohon: Pemohon Peninjauan Kembali dahulu Ter...,Direktori Putusan Mahkamah Agung Republik Indo...,putusan_1227_b_pk_pjk_2017_20250609095335,Pajak,NaN,...,MANUAL_DOWNLOAD,/content/drive/MyDrive/Penalaran Komputer/Penu...,putusan_1227_b_pk_pjk_2017_20250609095335.pdf,case_002_putusan_1227_b_pk_pjk_2017_2025060909...,16251,106,701,yang keliru dan telah mengabaikan fakta-fakta ...,NOT_IMPLEMENTED,Menolak permohonan
2,case_003,1236/B/PK/PJK/2017PUTUSANNomor 1236/B/PK/PJK/2...,5 Agustus 2016,: Direktori Putusan Mahkamah Agung Republik In...,ayat (1) undang-undang nomor; direktur jendera...,Pemohon: Pemohon Peninjauan Kembali dahuluTerb...,Direktori Putusan Mahkamah Agung Republik Indo...,putusan_1236_b_pk_pjk_2017_20250609100505,Pajak,NaN,...,MANUAL_DOWNLOAD,/content/drive/MyDrive/Penalaran Komputer/Penu...,putusan_1236_b_pk_pjk_2017_20250609100505.pdf,case_003_putusan_1236_b_pk_pjk_2017_2025060910...,14742,136,632,yang kelirudan telah mengabaikan fakta-fakta h...,NOT_IMPLEMENTED,Menolak permohonan
3,case_004,1245 B/PK/PJK/2017 PUTUSAN Nomor 1245/B/PK/PJK...,10 Agustus 2015,; Bahwa yang menjadi pokok sengketa dalam perm...,ayat (1) huruf d undang-undang nomor; ayat (1)...,Pemohon: Pemohon Peninjauan Kembali dahulu Ter...,Direktori Putusan Mahkamah Agung Republik Indo...,putusan_1245_b_pk_pjk_2017_20250609100435,Pajak,NaN,...,MANUAL_DOWNLOAD,/content/drive/MyDrive/Penalaran Komputer/Penu...,putusan_1245_b_pk_pjk_2017_20250609100435.pdf,case_004_putusan_1245_b_pk_pjk_2017_2025060910...,15100,123,674,yang keliru dan telah mengabaikan fakta-fakta ...,NOT_IMPLEMENTED,Menolak permohonan
4,case_005,1377/B/PK/Pjk/2019PUTUSANNomor 1377/B/PK/Pjk/2...,13 Agustus 2018,,huruf e undang-undang nomor,Pemohon: Pemohon Peninjauan Kembali; vs Termoh...,Direktori Putusan Mahkamah Agung Republik Indo...,putusan_1377_b_pk_pjk_2019_20250609100301,Pajak,NaN,...,MANUAL_DOWNLOAD,/content/drive/MyDrive/Penalaran Komputer/Penu...,putusan_1377_b_pk_pjk_2019_20250609100301.pdf,case_005_putusan_1377_b_pk_pjk_2019_2025060910...,1637,0,176,"Majelis Pengadilan Pajak,karena dalam perkara ...",NOT_IMPLEMENTED,Menolak permohonan



Columns in processed DataFrame: ['case_id', 'no_perkara', 'tanggal', 'ringkasan_fakta', 'pasal', 'pihak', 'text_full', 'judul_putusan', 'jenis_perkara', 'tingkat_proses', 'kata_kunci', 'tahun_dokumen', 'tanggal_register', 'lembaga_peradilan', 'amar_putusan', 'link_sumber', 'link_pdf', 'nama_file_pdf', 'nama_file_raw_text', 'jumlah_kata_full_text', 'jumlah_kata_ringkasan_fakta', 'jumlah_kata_argumen_hukum', 'argumen_hukum_utama', 'qa_pairs_sederhana', 'amar_kategori']
Shape of processed DataFrame: (43, 25)
[Tahap 2] Shape final DataFrame: (43, 25)
TAHAP 2: Case Representation — SELESAI

Tahap 2: Case Representation - Complete.
